# 99 · Cleanup

핸즈온 종료 후 비용 / 잔존 리소스 정리. **삭제는 비가역적** 이므로 한 셀씩 확인하면서 실행하세요.

In [ ]:
%run ./config

## 1. Serving Endpoint 삭제 (가장 비싼 리소스)

In [ ]:
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()

for name in [endpoint_basic, endpoint_wheel, endpoint_express, endpoint_torch_gpu]:
    try:
        w.serving_endpoints.delete(name=name)
        print(f"  ✓ deleted endpoint: {name}")
    except Exception as e:
        print(f"  - skip {name}: {e}")

## 2. UC 등록 모델 삭제 (선택)

In [ ]:
from mlflow import MlflowClient
client = MlflowClient()

models = [model_basic, model_pyfunc, model_deps, model_codepaths, model_wheel, model_express, model_torch_gpu]
for m in models:
    try:
        client.delete_registered_model(name=m)
        print(f"  ✓ deleted model: {m}")
    except Exception as e:
        print(f"  - skip {m}: {e}")

## 3. 테이블 삭제

In [ ]:
tables = [
    f"{catalog}.{schema}.customers",
    f"{catalog}.{schema}.churn_scores_batch",
]
for t in tables:
    spark.sql(f"DROP TABLE IF EXISTS {t}")
    print(f"  ✓ dropped {t}")

# Inference table들 — pattern 매칭
inf_tables = (
    spark.sql(f"SHOW TABLES IN {catalog}.{schema}")
         .filter("tableName LIKE '%_inference_payload' OR tableName LIKE '%_inference_request_logs'")
         .collect()
)
for row in inf_tables:
    full = f"{catalog}.{schema}.{row['tableName']}"
    spark.sql(f"DROP TABLE IF EXISTS {full}")
    print(f"  ✓ dropped inference table {full}")

## 4. Volume 비우기 (선택)

Volume 자체는 보존하고 내부 파일만 삭제하는 게 안전합니다.

In [ ]:
import os

for root, dirs, files in os.walk(volume_path, topdown=False):
    for f in files:
        os.remove(os.path.join(root, f))
    for d in dirs:
        try:
            os.rmdir(os.path.join(root, d))
        except OSError:
            pass

print(f"  ✓ cleaned {volume_path}")

## 5. (Optional) Schema 삭제

전체 schema 통째로 날릴 때.

In [ ]:
# 위험! 확인 후 주석 해제
# spark.sql(f"DROP SCHEMA IF EXISTS {catalog}.{schema} CASCADE")
# print(f"  ✓ dropped schema {catalog}.{schema}")

In [ ]:
print("✓ cleanup 완료")